# 13 · A linear baseline on the node features

The simplest model that can use what notebooks 11 and 12 built: a **regularised linear model** that reads each stay's own numbers and predicts the outcome. It never looks at the links, so it answers a clean question: *how much of the outcome can be read off a stay's description alone?* The graph neural networks have to beat this to justify using the network.

The network still reaches a linear model through the numbers computed from it (`n2v`, `structure`). That is what lets us ask whether **honest vs cheating networks** change what a linear model can learn, separately from **honest vs cheating diagnoses**.

The outcomes are yes/no, so the default is **logistic regression** (the standard linear model for a yes/no outcome). Set `MODEL = "ridge"` for plain linear regression on the 0/1 outcome instead; only the ranking of its scores matters for AUROC.

**Protocol, identical for every combination:** patient-grouped 60/20/20 train/validation/test splits (a patient's stays never straddle two sets); columns standardised with the training stays only; the regularisation strength chosen on the validation stays; the test stays scored once; all of it repeated over `SEEDS` split seeds and reported as mean and spread.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from ehrgraph.eval.aggregate import write_aggregate  # noqa: F401  (aggregate-only outputs, like notebooks 02/03)
from ehrgraph.gnn.linear_probe import YEAR_BANDS, paired, run_probes, summarize, evaluate_arrays
from ehrgraph.gnn.node_features import FEATURE_SETS, LEAKABLE_BLOCKS
from ehrgraph.project import setup
from ehrgraph.viz import network_figures as nf

p = setup()
nf.apply_style()

TASKS = {"guo_los": "long stay", "new_hypertension": "new high blood pressure"}
NAME = {"safe": "honest", "leaky": "cheating", "null": "random"}
MODEL = "logistic"             # or "ridge"
SEEDS = range(10)              # split seeds; a few seconds each

## Load the inputs from notebook 12

`gnn_inputs_<task>.pt` holds one `Data` per combination. Stays are in the same order as the labels, so the **prediction year** (not stored in the file) can be rebuilt from the labels and checked against the saved outcomes and patients. A mismatch would stop the notebook rather than silently mislabel stays.

In [ ]:
datasets, years = {}, {}
for task in TASKS:
    datasets[task] = torch.load(p.results / f"gnn_inputs_{task}.pt", weights_only=False)

    lab = p.labels(task)
    lab = lab[lab["patient_id"].isin(set(p.diagnoses))].reset_index(drop=True)     # same filter as the graph builder
    d0 = next(iter(datasets[task].values()))
    assert (lab["value"].astype(int).to_numpy() == d0.y.numpy()).all(), "labels are not aligned with the saved stays"
    assert (pd.factorize(lab["patient_id"])[0] == d0.patient_idx.numpy()).all(), "patients are not aligned"
    years[task] = pd.to_datetime(lab["prediction_time"]).dt.year.to_numpy()

    d_n = datasets[task].get(("safe", "safe", "n2v"))
    width = len(d_n.feature_names) - 1 if d_n is not None else "n/a"        # minus the "no links" flag column
    print(f"{TASKS[task]}: {d0.num_nodes:,} stays, {d0.y.float().mean():.1%} positive, "
          f"{len(set(d0.patient_idx.tolist())):,} patients, node2vec width {width}, "
          f"{len(datasets[task])} saved combinations")

## Which combinations are worth running

A linear model only sees the numbers, so some saved combinations are identical:

- `dx` and `clmbr` don't use the network, so the network label is irrelevant: run them once.
- `structure` and `n2v` don't use the diagnosis version, so only the honest-features copy is run.
- `dx_n2v` and `clmbr_n2v` mix both, so every network matters.

In [ ]:
GRAPH_BLOCKS = {"structure", "rwpe", "n2v"}
needs_graph = lambda fs: any(b in GRAPH_BLOCKS for b in FEATURE_SETS[fs])
uses_dx = lambda fs: any(b in LEAKABLE_BLOCKS for b in FEATURE_SETS[fs])

def pretty(key):
    g, s, fs = key
    return " | ".join([fs] + ([f"net: {NAME[g]}"] if needs_graph(fs) else []) + ([f"feat: {NAME[s]}"] if uses_dx(fs) else []))

keys = {t: [k for k in ds if needs_graph(k[2]) or k[0] == "safe"] for t, ds in datasets.items()}
for t in TASKS:
    print(TASKS[t], len(keys[t]), "runs:", [pretty(k) for k in keys[t]])

## Run

Each combination is fitted on `len(SEEDS)` different patient-grouped splits.

In [ ]:
res = {t: run_probes(datasets[t], keys[t], SEEDS, MODEL, year=years[t], log=lambda s: None) for t in TASKS}

tables = {}
for t in TASKS:
    s = summarize(res[t])
    s.index = [pretty(k) for k in s.index]
    tables[t] = s
    print(f"{TASKS[t]} - test AUROC, mean and spread over {len(SEEDS)} splits ({MODEL})")
    display(s[["mean", "sd", "test_ap", "n_features"]].round(3))

### What do trivial models score?

The prediction **year** alone (see notebook 11: the blood-pressure rate falls about fourfold across years), and the share of stays with the outcome (average precision of a model that knows nothing equals this share).

In [ ]:
rows = []
for t in TASKS:
    y_t = datasets[t][keys[t][0]].y.numpy(); g_t = datasets[t][keys[t][0]].patient_idx.numpy()
    yr = years[t].astype(float)[:, None]
    r = pd.DataFrame([evaluate_arrays(yr, np.array([True]), y_t, g_t, seed, MODEL) for seed in SEEDS])
    rows.append({"question": TASKS[t], "year alone, test AUROC": r.test_auc.mean(),
                 "spread": r.test_auc.std(), "outcome share (the AP of knowing nothing)": y_t.mean()})
display(pd.DataFrame(rows).set_index("question").round(3))

## The questions we care about

Each row is a **paired difference** in test AUROC (A minus B), taken on identical splits, so split luck cancels. `fraction_a_higher` is the share of splits where A won. Read the mean difference against its spread; a difference smaller than the spread is not a finding.

In [ ]:
PAIRS = [
    ("cheating diagnoses vs honest (dx, honest net)",          ("safe", "leaky", "dx"),            ("safe", "safe", "dx")),
    ("cheating network vs honest (n2v)",                       ("leaky", "safe", "n2v"),           ("safe", "safe", "n2v")),
    ("cheating network vs honest (structure)",                 ("leaky", "safe", "structure"),     ("safe", "safe", "structure")),
    ("cheating network vs honest (dx + n2v, honest diagnoses)", ("leaky", "safe", "dx_n2v"),        ("safe", "safe", "dx_n2v")),
    ("both cheat vs both honest (dx + n2v)",                   ("leaky", "leaky", "dx_n2v"),       ("safe", "safe", "dx_n2v")),
    ("real network vs random network (n2v)",                   ("safe", "safe", "n2v"),            ("null", "safe", "n2v")),
    ("does n2v add to diagnoses? (honest)",                    ("safe", "safe", "dx_n2v"),         ("safe", "safe", "dx")),
    ("CLMBR vs diagnoses (honest)",                            ("safe", "safe", "clmbr"),          ("safe", "safe", "dx")),
    ("does n2v add to CLMBR?",                                 ("safe", "safe", "clmbr_n2v"),      ("safe", "safe", "clmbr")),
]
out = []
for t in TASKS:
    for name, a, b in PAIRS:
        if a in datasets[t] and b in datasets[t]:
            out.append({"question": TASKS[t], "A minus B": name, **paired(res[t], a, b)})
display(pd.DataFrame(out).set_index(["question", "A minus B"]).round(3))

## Year-adjusted view

Test AUROC **within** calendar-year bands (2015 is the coding switch). A model that wins only by learning the calendar will not keep its score here. Bands with fewer than 10 positive or 10 negative test stays are left blank, so blood-pressure bands are noisy.

In [ ]:
band_cols = [f"auc_{b}" for b in YEAR_BANDS]
for t in TASKS:
    df = res[t].copy(); df["model"] = [pretty((g, s, f)) for g, s, f in zip(df.graph, df.source, df.feature_set)]
    print(TASKS[t]); display(df.groupby("model")[["test_auc", *band_cols]].mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, len(TASKS), figsize=(13, 4.6), sharex=False)
for ax, t in zip(np.atleast_1d(axes), TASKS):
    s = tables[t].sort_values("mean")
    ax.barh(s.index, s["mean"], xerr=s["sd"], color="#2b7bba", ecolor="#444", capsize=2)
    ax.axvline(0.5, color="grey", lw=0.8, ls=":"); ax.set_xlim(0.4, 1.0)
    ax.set_title(TASKS[t]); ax.set_xlabel(f"test AUROC ({MODEL}, mean ± spread over {len(SEEDS)} splits)")
plt.tight_layout()
nf.save(fig, p.figures / "13_linear_baselines.png")
plt.show()

In [ ]:
# aggregate numbers only (no patient data)
pd.concat({TASKS[t]: res[t] for t in TASKS}, names=["question"]).reset_index(level=0) \
  .to_csv(p.results / "linear_baselines.csv", index=False)

## What we learned

*(fill in after running)*

- **Which description predicts best, per question?** Is it the same for long stay and blood pressure?
- **Does cheating show up in the linear model?** Blood pressure should jump with cheating diagnoses (the future record contains the diagnosis, notebook 03); long stay should not (notebook 02).
- **Does the cheating network change what node2vec can do?** Compare the two `n2v` rows against their spread.
- **Does node2vec add anything beyond diagnoses?** The `dx + n2v` vs `dx` row. Notebook 12's tree results said no.
- **Year:** does any model's score survive within year bands, or does it collapse towards the year-only score?
- **CLMBR vs diagnoses** (only if `CLMBR_PATH` was set in notebook 12).

## Read these results carefully

- **Node2vec is transductive.** It was trained on the whole network, test stays included. No outcomes were used, so it is not label leakage, but test stays' embeddings were shaped by links to training stays.
- **Possible time leak in same-patient links.** A link to a patient's *later* stay brings in that stay's record. The `structure` and `n2v` rows may carry this; the check from notebook 12 (earlier vs later same-patient links) decides it. If it is confirmed, these rows are not fully honest and a time-aware network is needed.
- **Calendar year.** See the year-alone score and the year-band table above.
- **This model ignores the links.** A GNN that beats it is using the network; one that doesn't is not.
- **Later:** the EHRSHOT few-shot setting (a handful of labelled stays), and GCN / GraphSAGE / GAT on the same splits (use `make_splits` from `linear_probe` so they share the split seeds).